In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.nn import GINConv, global_mean_pool
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_curve, auc, confusion_matrix,
    ConfusionMatrixDisplay, precision_score,
    recall_score, f1_score
)
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import numpy as np
import random

# Reproducibility
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
torch.cuda.manual_seed_all(seed)

# === Load Dataset ===
def load_pytorch_dataset(file_path):
    with open(file_path, 'rb') as f:
        dataset = pickle.load(f)
    print(f"✅ Dataset successfully loaded from {file_path}")
    return dataset

pickle_file_path = r"C:\Users\nidhi\Downloads\Pickle files\Random Split\One-Hot\Class A One-Hot encode.pkl"
pytorch_dataset = load_pytorch_dataset(pickle_file_path)

# === Model ===
class SimpleGINModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout_rate):
        super(SimpleGINModel, self).__init__()
        self.gin1 = GINConv(nn.Linear(input_dim, hidden_dim))
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.gin2 = GINConv(nn.Linear(hidden_dim, hidden_dim))
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.gin3 = GINConv(nn.Linear(hidden_dim, hidden_dim))
        self.bn3 = nn.BatchNorm1d(hidden_dim)
        self.gin4 = GINConv(nn.Linear(hidden_dim, hidden_dim))
        self.bn4 = nn.BatchNorm1d(hidden_dim)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.bn1(self.gin1(x, edge_index)))
        x = F.relu(self.bn2(self.gin2(x, edge_index)))
        x = F.relu(self.bn3(self.gin3(x, edge_index)))
        x = F.relu(self.bn4(self.gin4(x, edge_index)))
        x = self.dropout(x)
        x = global_mean_pool(x, batch)
        return self.fc(x)

# === Training ===
def train(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct = 0, 0
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        data.y = data.y.long()
        out = model(data)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
        correct += (out.argmax(dim=1) == data.y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

# === Evaluation ===
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0, 0
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            data.y = data.y.long()
            out = model(data)
            loss = criterion(out, data.y)
            total_loss += loss.item() * data.num_graphs
            correct += (out.argmax(dim=1) == data.y).sum().item()
            probs = F.softmax(out, dim=1)[:, 1].cpu().numpy()
            all_labels.extend(data.y.cpu().numpy())
            all_preds.extend(out.argmax(dim=1).cpu().numpy())
            all_probs.extend(probs)
    return total_loss / len(loader.dataset), correct / len(loader.dataset), \
           np.array(all_labels), np.array(all_preds), np.array(all_probs)

# === Plot Training Metrics (Without Labels & Legend) ===
def plot_metrics(epochs, train_loss, train_acc, test_loss, test_acc, model_type):
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, train_loss, color="red")
    plt.plot(epochs, test_loss, color="orange")
    plt.plot(epochs, train_acc, color="green")
    plt.plot(epochs, test_acc, color="blue")
    plt.xticks([])
    plt.yticks([])
    plt.title(f"{model_type} Metrics", fontsize=14)
    plt.tight_layout()
    plt.savefig(f"{model_type}_metrics_plot.png", bbox_inches='tight')
    plt.close()
    print(f"🖼️ Metrics plot saved as {model_type}_metrics_plot.png")

# === Prepare Data ===
train_data, test_data = train_test_split(pytorch_dataset, test_size=0.2, random_state=seed)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

# === Initialize Model ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleGINModel(20, 64, 2, 0.5).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

# === Training Loop ===
num_epochs = 350
train_losses, train_accs, test_losses, test_accs = [], [], [], []
history = []

for epoch in range(1, num_epochs + 1):
    tr_loss, tr_acc = train(model, train_loader, optimizer, criterion, device)
    te_loss, te_acc, y_true, y_pred, y_probs = evaluate(model, test_loader, criterion, device)

    train_losses.append(tr_loss)
    train_accs.append(tr_acc)
    test_losses.append(te_loss)
    test_accs.append(te_acc)
    history.append((y_true, y_pred, y_probs))

    print(f"Epoch {epoch:03d}: Train Loss={tr_loss:.4f}, Train Acc={tr_acc:.4f}, Test Loss={te_loss:.4f}, Test Acc={te_acc:.4f}")

# === Final Plot ===
plot_metrics(range(1, num_epochs + 1),
             train_losses, train_accs,
             test_losses, test_accs,
             "A_OneHot_64_NoWD"
)

# === Save Epoch-wise Metrics ===
df_metrics = pd.DataFrame({
    "Epoch": list(range(1, num_epochs + 1)),
    "Train Accuracy": [round(acc * 100, 2) for acc in train_accs],
    "Train Loss": [round(loss, 2) for loss in train_losses],
    "Test Accuracy": [round(acc * 100, 2) for acc in test_accs],
    "Test Loss": [round(loss, 2) for loss in test_losses],
})

df_metrics.to_excel("A_OneHot_64_NoWD.xlsx", index=False)
print("📊 Epoch-wise metrics saved to 'A_OneHot_32_NoWD.xlsx'")

✅ Dataset successfully loaded from C:\Users\nidhi\Downloads\Pickle files\Random Split\One-Hot\Class A One-Hot encode.pkl
Epoch 001: Train Loss=0.6974, Train Acc=0.5023, Test Loss=0.6714, Test Acc=0.6564
Epoch 002: Train Loss=0.6766, Train Acc=0.6379, Test Loss=0.6573, Test Acc=0.6564
Epoch 003: Train Loss=0.6565, Train Acc=0.7257, Test Loss=0.6515, Test Acc=0.6564
Epoch 004: Train Loss=0.6392, Train Acc=0.7411, Test Loss=0.6442, Test Acc=0.7301
Epoch 005: Train Loss=0.6231, Train Acc=0.7504, Test Loss=0.6340, Test Acc=0.7301
Epoch 006: Train Loss=0.6092, Train Acc=0.7504, Test Loss=0.6227, Test Acc=0.7301
Epoch 007: Train Loss=0.5964, Train Acc=0.7458, Test Loss=0.6130, Test Acc=0.7301
Epoch 008: Train Loss=0.5835, Train Acc=0.7488, Test Loss=0.6031, Test Acc=0.7301
Epoch 009: Train Loss=0.5714, Train Acc=0.7550, Test Loss=0.5931, Test Acc=0.7301
Epoch 010: Train Loss=0.5607, Train Acc=0.7519, Test Loss=0.5836, Test Acc=0.7301
Epoch 011: Train Loss=0.5505, Train Acc=0.7535, Test Loss=0